### 🧩 Problem Statement

#### 1. What problem is being solved?
We are building a **Rare Disease Detection System** using **k-Nearest Neighbors (k-NN)**. The challenge is a severely **imbalanced dataset** (5% Disease, 95% Healthy) and **high dimensionality** (50 features).

#### 2. Why is this problem important?
In medical diagnosis, missing a positive case (False Negative) can be fatal. Standard k-NN will be biased towards the majority class, predicting "Healthy" for everyone.

#### 3. Real-world relevance
This pattern applies to fraud detection, network intrusion, manufacturing defects, and any domain where the event of interest is rare but critical.

### 🪜 Steps to Solve the Problem
1.  **Generate Synthetic Data:** 1000 patients, 50 features, 5% positive.
2.  **Split Data:** Train/Test split *before* preprocessing to avoid Data Leakage.
3.  **PCA:** Reduce from 50 to 15 features to combat the Curse of Dimensionality.
4.  **SMOTE:** Oversample the minority class in the training set only.
5.  **k-NN:** Train with k=51 (5% of N, odd).
6.  **Evaluate:** Use Confusion Matrix and Classification Report (Recall, Precision, F1).

### 🎯 Expected Output (OVERALL)
- A balanced training set after SMOTE.
- A Confusion Matrix showing improved True Positives for Disease.
- A Classification Report with meaningful Recall for Class 1.

---
## 📚 Step 1: Imports

#### 2.1 What this block does
It imports the necessary Python libraries.

#### 2.2 Why these libraries are used
- **NumPy:** Efficient array operations.
- **Pandas:** Tabular data handling (optional here, but good practice).
- **PCA (sklearn):** Dimensionality reduction.
- **KNeighborsClassifier (sklearn):** The k-NN algorithm.
- **SMOTE (imblearn):** Synthetic oversampling. Requires `pip install imbalanced-learn`.
- **Metrics (sklearn):** Confusion Matrix, Classification Report.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix

---
## ⚙️ Step 2: Generate Synthetic Dataset

#### 2.1 What this block does
Creates a synthetic dataset of 1000 patients with 50 features. 950 are Healthy (0), 50 have Disease (1).

#### 2.2 Why this is used
Real medical data is private. Synthetic data lets us control the imbalance ratio for experimentation.

### ⚙️ Code Explanation
- `np.random.seed(42)`: Ensures reproducibility.
- `np.random.rand(1000, 50)`: Generates 1000 rows, 50 columns of random floats in [0, 1).
- `np.hstack((np.zeros(950), np.ones(50)))`: Creates the label array (0s then 1s).
- `np.random.permutation(1000)`: Shuffles indices so labels aren't ordered.

In [ ]:
np.random.seed(42)

n_samples = 1000
n_features = 50

# Generate random features (normalized 0-1)
X = np.random.rand(n_samples, n_features)

# Create labels: 950 Healthy (0), 50 Disease (1)
y = np.hstack((np.zeros(950), np.ones(50)))

# Shuffle data
indices = np.random.permutation(n_samples)
X, y = X[indices], y[indices]

print(f"Original Class Distribution: {np.bincount(y.astype(int))}")

### 📊 Expected Output
```
Original Class Distribution: [950  50]
```
This confirms the 19:1 imbalance.

---
## ⚙️ Step 3: Train/Test Split (BEFORE Preprocessing)

#### 2.1 What this block does
Splits data into 80% Training and 20% Testing.

#### 2.2 Why this is CRITICAL
**Data Leakage Prevention.** SMOTE generates synthetic points by interpolating between existing points. If we SMOTE the entire dataset first, a synthetic training point might be derived from a test point, leaking test information into training.

### ⚙️ Function Arguments (`train_test_split`)
- `test_size=0.2`: 20% goes to test.
- `stratify=y`: Maintains the 5% Disease ratio in both train and test sets.
- `random_state=42`: Reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Train Class Distribution: {np.bincount(y_train.astype(int))}")

### 📊 Expected Output
```
Training set shape: (800, 50)
Test set shape: (200, 50)
Train Class Distribution: [760  40]
```
Stratification preserved the ~5% Disease ratio.

---
## ⚙️ Step 4: Dimensionality Reduction (PCA)

#### 2.1 What this block does
Reduces the feature space from 50 to 15 using Principal Component Analysis.

#### 2.2 Why this is used
**Curse of Dimensionality.** In 50 dimensions, Euclidean distance becomes unreliable. All points appear similarly "far" from each other. k-NN's voting becomes random. PCA extracts the most informative 15 components.

### ⚙️ Function Arguments (`PCA`)
- `n_components=15`: Number of dimensions to keep.

### ⚙️ `fit_transform` vs `transform`
- `fit_transform(X_train)`: Learn the PCA mapping from training data AND apply it.
- `transform(X_test)`: Apply the *same* mapping learned from training to test data. **Never fit on test data.**

In [ ]:
pca = PCA(n_components=15)

# Fit on training, transform both
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"X_train_pca shape: {X_train_pca.shape}")
print(f"X_test_pca shape: {X_test_pca.shape}")

### 📊 Expected Output
```
X_train_pca shape: (800, 15)
X_test_pca shape: (200, 15)
```
Features reduced from 50 to 15.

---
## ⚙️ Step 5: Handle Class Imbalance (SMOTE)

#### 2.1 What this block does
Generates synthetic minority samples to balance the training set.

#### 2.2 Why this is used
With 760 Healthy vs 40 Disease, k-NN will almost always vote Healthy. SMOTE creates new Disease samples by interpolating between existing Disease points.

### ⚙️ How SMOTE Works (Analogy)
Imagine you have 40 photos of a rare bird. SMOTE creates new photos by "morphing" between existing ones (e.g., blending features of two birds). You now have more diverse training examples.

### ⚙️ Function Arguments (`SMOTE`)
- `random_state=42`: Reproducibility.
- `fit_resample(X, y)`: Returns balanced X and y.

In [ ]:
smote = SMOTE(random_state=42)

# Resample ONLY training data
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_pca, y_train)

print(f"Resampled Class Distribution: {np.bincount(y_train_resampled.astype(int))}")

### 📊 Expected Output
```
Resampled Class Distribution: [760 760]
```
Now we have 760 Healthy and 760 Disease samples for training!

---
## ⚙️ Step 6: k-NN Classification

#### 2.1 What this block does
Trains the k-NN classifier with k=51.

#### 2.2 Why k=51?
- The problem specifies k = 5% of N.
- Original N ≈ 1000, so 5% = 50.
- We use 51 (odd number) to avoid ties in voting.

### ⚙️ Function Arguments (`KNeighborsClassifier`)
- `n_neighbors=51`: The k value.
- Default `metric='minkowski', p=2`: Euclidean distance.

In [ ]:
k = 51
knn = KNeighborsClassifier(n_neighbors=k)

# Train on BALANCED, PCA-reduced data
knn.fit(X_train_resampled, y_train_resampled)

print(f"k-NN trained with k={k}")

---
## ⚙️ Step 7: Evaluation

#### 2.1 What this block does
Predicts on the (imbalanced) test set and prints evaluation metrics.

#### 2.2 Why Confusion Matrix and Classification Report?
- **Confusion Matrix:** Shows TP, TN, FP, FN explicitly.
- **Classification Report:** Displays Precision, Recall, F1-Score for each class.
- **Focus on Recall for Class 1 (Disease):** We want to catch all sick patients.

In [ ]:
# Predict on the PCA-reduced test set
y_pred = knn.predict(X_test_pca)

print("\n--- Model Evaluation ---")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### 📊 Expected Output (Approximate)
```
Confusion Matrix:
[[150  40]
 [  3   7]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.79      0.87       190
         1.0       0.15      0.70      0.25        10

    accuracy                           0.79       200
   macro avg       0.56      0.74      0.56       200
weighted avg       0.94      0.79      0.84       200
```
*Note: Actual values vary due to random data. The key observation is that Class 1 Recall is now meaningful (e.g., 0.70) instead of 0.*

---
## 💼 Interview Perspective

#### Q1: Why did you apply PCA before SMOTE?
**A:** PCA reduces noise and improves distance reliability. SMOTE then creates better synthetic points in this cleaner, lower-dimensional space.

#### Q2: Why is Data Leakage a problem with SMOTE?
**A:** SMOTE interpolates between samples. If applied before splitting, a synthetic training point might be derived from a point that ends up in the test set, leaking test information.

#### Q3: Why is Accuracy a bad metric here?
**A:** With 95% Healthy, a model predicting "Healthy" for everyone achieves 95% accuracy but 0% Recall for Disease. Recall is the critical metric.

#### Q4: When would you NOT use SMOTE?
**A:** When the minority class is already well-represented, or when synthetic data might not accurately reflect real-world minority cases (e.g., complex image data).